**Logistic regression**

In [1]:
import os
# Limit OS-level process affinity to exclusively use Core 0.
# This strictly physically constraints Polars, PyTorch, and everything else in this notebook to 1 core.
os.sched_setaffinity(0, {2,3})

# Also keep your GPU and general thread limits just in case
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["POLARS_MAX_THREADS"] = "2" # Must be set before 'import polars'

In [1]:
import polars as pl
import importlib
import lr_rf_cuml_based
from datetime import date, datetime
import json

/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/rapids_dask_dependency/dask_loader.py:36: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  return importlib.import_module(spec.name)


In [2]:
importlib.reload(lr_rf_cuml_based)
from lr_rf_cuml_based import mach_lern

In [ ]:
df_aapl = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/AAPL.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_msft = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/MSFT.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_googl = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/GOOGL.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_amzn = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/AMZN.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect()
df_nvda = pl.scan_csv('/mnt/windows/windows_hanka_bcthesis/full_stock_prices/NVDA.csv').with_columns(pl.col('date').str.to_date('%Y-%m-%d')).collect() # Or FB.csv

# Package them up
dict_of_dfs = {
    "AAPL": df_aapl.select(['date', 'close']),
    "MSFT": df_msft.select(['date', 'close']),
    "GOOGL": df_googl.select(['date', 'close']),
    "AMZN": df_amzn.select(['date', 'close']),
    "NVDA": df_nvda.select(['date', 'close'])
}

df_tfidf = (
    pl.scan_parquet('/mnt/windows/windows_hanka_bcthesis/full_news/tfidf_nasdaq.parquet')
    # Use whatever the date column is actually called in this file
    .filter(pl.col("trading_session_date_utc").is_between(pl.date(2006, 10, 20), pl.date(2019, 12, 31)))
    .collect()
)

df_sent = (
    pl.scan_parquet('/mnt/red/red_hanka_bcthesis/full_news/finbert_nasdaq_2006-2023_avg_sentiment.parquet')
    # Use whatever the date column is actually called in this file
    .filter(pl.col("trading_session_date_utc").is_between(pl.date(2006, 10, 20), pl.date(2019,12, 31)))
    .collect()
)

In [7]:
ml_object = mach_lern(test_size=0.3)

In [ ]:
ml_object.load_and_prepare_multiple_price_data(dict_of_dfs, start_date=date(2006, 10, 20), end_date=date(2019, 12, 31)) 
ml_object.load_tfidf_data(df_tfidf)
final_df = ml_object.load_finbert_data(df_sent)

Loading prices for multiple stocks: ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']...
Master multi-stock timeline established. Shape: (16570, 9)


trading_date_utc,nyse_ticker,target_next_day,return_lag0,return_lag1,return_lag2,return_lag3,return_lag4,return_lag5
date,str,i8,f64,f64,f64,f64,f64,f64
2006-10-30,"""AAPL""",1,0.000124,-0.021657,0.006244,0.007773,-0.005033,0.018887
2006-10-30,"""MSFT""",1,0.006704,-0.000353,0.001413,0.001061,-0.005975,0.000703
2006-10-30,"""GOOGL""",0,0.002883,-0.020408,-0.003083,0.028079,-0.015537,0.045924
2006-10-30,"""AMZN""",0,-0.002354,-0.001567,0.016454,0.120428,0.02281,0.009518
2006-10-30,"""NVDA""",1,0.012044,-0.043144,0.031707,0.009231,0.029133,-0.006606
…,…,…,…,…,…,…,…,…
2019-12-30,"""AAPL""",1,0.005935,-0.000379,0.01984,0.000951,0.016318,-0.002071
2019-12-30,"""MSFT""",1,-0.008619,0.001828,0.008197,-0.000191,0.0,0.010918
2019-12-30,"""GOOGL""",0,-0.011021,-0.005747,0.013418,-0.00459,-0.000437,-0.003848


In [14]:
# Start the engine manually. 
# It will print the dashboard link ONE time, and that link will stay valid.
ml_object.start_dask_cluster()

Booting up LocalCUDACluster...


/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/rapids_dask_dependency/dask_loader.py:36: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  return importlib.import_module(spec.name)
/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/rapids_dask_dependency/dask_loader.py:36: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  return importlib.import_module(spec.name)


Dask dashboard available at: http://127.0.0.1:8787/status


In [ ]:
try:
    with open("optimal_parameters.jsonl", "r") as f:
        lines = f.readlines()

    # filter for LR entries
    filtered_entries = []
    for line in lines:
        entry = json.loads(line)
        if entry.get("model_type") == "LR":
            ts = datetime.strptime(entry["timestamp"], "%Y-%m-%d %H:%M:%S")
            if ts >= datetime(2026, 3, 10, 21, 0, 0) and ts <= datetime(2026, 3, 10, 23, 59, 59):
                filtered_entries.append(entry)

    # evaluate each set of parameters
    nmb = 1
    for entry in filtered_entries:
        mode = entry["mode"]
        param_grid = entry["hyperparameters"]
        param_grid = {k: [v] for k, v in param_grid.items()}
        acc = entry["best_cv_accuracy"]
        auc = entry["best_cv_auc"]
        print(f"\n{nmb }. Evaluating LR mode: {mode} with params: {param_grid}")
        print(f"\nTrain accuracy {acc} and AUC: {auc}")

        best_pipeline, x_test, y_test, active_features = ml_object.train_logistic_regression(mode=mode, param_grid=param_grid)
        y_pred, y_probs = ml_object.evaluate(
            mode=mode, 
            best_pipeline=best_pipeline, 
            x_test=x_test, 
            y_test=y_test, 
            active_features=active_features
        )
        nmb += 1
finally:
    ml_object.stop_dask_cluster()


1. Evaluating LR mode: price with params: {'logr__C': [0.01], 'logr__class_weight': [None], 'logr__l1_ratio': [1.0], 'logr__tol': [0.0001]}

Train accuracy 0.5102 and AUC: 0.5024

Starting Experiment: PRICE (6 features)

--- Running GPU GridSearchCV (5 splits) ---
[2026-03-11 19:04:15.986] [CUML] [warning] QWL-QN stopped, because the line search failed to advance (step delta = 0.000000)
[2026-03-11 19:04:16.099] [CUML] [warning] QWL-QN stopped, because the line search failed to advance (step delta = 0.000000)
Calculating CV AUC for best model...
Best Params: {'logr__C': 0.01, 'logr__class_weight': None, 'logr__l1_ratio': 1.0, 'logr__tol': 0.0001}
Best CV Accuracy: 0.5102 | Best CV AUC: 0.5024
Optimal parameters saved to optimal_parameters.jsonl

--- Final Model Reality Check (Untouched Test Set) ---
Min Prob: 0.4785, Max Prob: 0.5259, Mean Prob: 0.5075
Final Test Accuracy: 0.5406
Final Test ROC AUC: 0.5233

Classification Report:
              precision    recall  f1-score   support



/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.28 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


[2026-03-11 19:04:40.481] [CUML] [warning] QWL-QN: max iterations reached
[2026-03-11 19:04:40.482] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.


/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.28 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


[2026-03-11 19:05:09.782] [CUML] [warning] QWL-QN: max iterations reached
[2026-03-11 19:05:09.782] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
Calculating CV AUC for best model...
[2026-03-11 19:05:38.529] [CUML] [warning] QWL-QN: max iterations reached
[2026-03-11 19:05:38.530] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.
Best Params: {'logr__C': 0.3, 'logr__class_weight': None, 'logr__l1_ratio': 0.8, 'logr__tol': 0.0001}
Best CV Accuracy: 0.5025 | Best CV AUC: 0.4957
Optimal parameters saved to optimal_parameters.jsonl

--- Final Model Reality Check (Untouched Test Set) ---
Min Prob: 0.0000, Max Prob: 1.0000, Mean Prob: 0.4328
Final Test Accuracy: 0.4926
Final Test ROC AUC: 0.5025

Classif

/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Calculating CV AUC for best model...
Best Params: {'logr__C': 0.01, 'logr__class_weight': 'balanced', 'logr__l1_ratio': 0.0, 'logr__tol': 0.001}
Best CV Accuracy: 0.5054 | Best CV AUC: 0.5004
Optimal parameters saved to optimal_parameters.jsonl

--- Final Model Reality Check (Untouched Test Set) ---
Min Prob: 0.0000, Max Prob: 1.0000, Mean Prob: 0.6662
Final Test Accuracy: 0.5104
Final Test ROC AUC: 0.5039

Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.45      0.33      0.38      2459
      Up (1)       0.54      0.66      0.59      2896

    accuracy                           0.51      5355
   macro avg       0.50      0.50      0.49      5355
weighted avg       0.50      0.51      0.50      5355


--- Weight/Importance Analysis ---
Metric: Linear Coefficients (Magnitude = Impact, Sign = Direction)
--- Weight Analysis ---
Average Absolute Price Weight (6 features): 0.030246
Average Absolute TF-IDF Importance (5000 features): 0.02418

**Random Forest**

In [15]:
#Evaluate best RF parameters on test data
with open("optimal_parameters.jsonl", "r") as f:
    lines = f.readlines()

# filter for RF entries
filtered_entries = []
for line in lines:
    entry = json.loads(line)
    if entry.get("model_type") == "RF":
        ts = datetime.strptime(entry["timestamp"], "%Y-%m-%d %H:%M:%S")
        if ts >= datetime(2026, 3, 10, 21, 0, 0) and ts <= datetime(2026, 3, 11, 1, 59, 59):
            filtered_entries.append(entry)

# evaluate each set of parameters
nmb = 1
for entry in filtered_entries:
    mode = entry["mode"]
    param_grid = entry["hyperparameters"]
    param_grid = {k: [v] for k, v in param_grid.items()}
    acc = entry["best_cv_accuracy"]
    auc = entry["best_cv_auc"]
    print(f"\n{nmb }. Evaluating RF mode: {mode} with params: {param_grid}")
    print(f"\nTrain accuracy {acc} and AUC: {auc}")

    best_pipeline, x_test, y_test, active_features = ml_object.train_random_forest(mode=mode, param_grid=param_grid)
    y_pred, y_probs = ml_object.evaluate(
        mode=mode, 
        best_pipeline=best_pipeline, 
        x_test=x_test, 
        y_test=y_test, 
        active_features=active_features
    )
    nmb += 1


1. Evaluating RF mode: price with params: {'rf__max_depth': [14], 'rf__max_features': ['sqrt'], 'rf__min_samples_leaf': [10], 'rf__n_estimators': [400]}

Train accuracy 0.5071 and AUC: 0.5008

Starting Experiment: RF PRICE (6 features)

--- Running GPU GridSearchCV (5 splits) ---
Calculating CV AUC for best model...
Best Params: {'rf__max_depth': 14, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 10, 'rf__n_estimators': 400}
Best CV Accuracy: 0.5071 | Best CV AUC: 0.5008
Optimal parameters saved to optimal_parameters.jsonl

--- Final Model Reality Check (Untouched Test Set) ---
Min Prob: 0.2261, Max Prob: 0.6744, Mean Prob: 0.5099
Final Test Accuracy: 0.5154
Final Test ROC AUC: 0.5154

Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.47      0.40      0.43      2459
      Up (1)       0.55      0.61      0.58      2896

    accuracy                           0.52      5355
   macro avg       0.51      0.51      0.51      5355
weig

/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.28 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.28 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Calculating CV AUC for best model...
Best Params: {'rf__max_depth': 6, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 10, 'rf__n_estimators': 200}
Best CV Accuracy: 0.5159 | Best CV AUC: 0.4995
Optimal parameters saved to optimal_parameters.jsonl

--- Final Model Reality Check (Untouched Test Set) ---
Min Prob: 0.4534, Max Prob: 0.5433, Mean Prob: 0.4995
Final Test Accuracy: 0.5154
Final Test ROC AUC: 0.5112

Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.47      0.52      0.50      2459
      Up (1)       0.56      0.51      0.53      2896

    accuracy                           0.52      5355
   macro avg       0.52      0.52      0.51      5355
weighted avg       0.52      0.52      0.52      5355


--- Weight/Importance Analysis ---
Metric: Gini Importance (Magnitude = How often it was useful)
--- Weight Analysis ---
Price Features: [None in this mode]
Average Absolute TF-IDF Importance (5000 features): 0.000200
Most influent

/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/home/sj5/.micromamba/envs/thesis_ml/lib/python3.10/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 477.85 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Calculating CV AUC for best model...
Best Params: {'rf__max_depth': 6, 'rf__max_features': 'log2', 'rf__min_samples_leaf': 10, 'rf__n_estimators': 200}
Best CV Accuracy: 0.5128 | Best CV AUC: 0.5106
Optimal parameters saved to optimal_parameters.jsonl

--- Final Model Reality Check (Untouched Test Set) ---
Min Prob: 0.4659, Max Prob: 0.5363, Mean Prob: 0.5018
Final Test Accuracy: 0.5156
Final Test ROC AUC: 0.5047

Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.47      0.42      0.45      2459
      Up (1)       0.55      0.59      0.57      2896

    accuracy                           0.52      5355
   macro avg       0.51      0.51      0.51      5355
weighted avg       0.51      0.52      0.51      5355


--- Weight/Importance Analysis ---
Metric: Gini Importance (Magnitude = How often it was useful)
--- Weight Analysis ---
Average Absolute Price Weight (6 features): 0.000191
Average Absolute TF-IDF Importance (5000 features): 0.000